In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import sys

In [2]:
sys.argv = ['-f'] + ["MIC", "-a", "cpu"]

In [3]:
import os
sys.path.append("/home/pwiesenbach/BertGCN")
os.chdir("/home/pwiesenbach/BertGCN")

In [4]:
from entry import * 

/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import importlib
import entry
import logging
importlib.reload(logging)
importlib.reload(entry)

<module 'entry' from '/beegfs/homes/pwiesenbach/BertGCN/entry.py'>

In [8]:
from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer
from operator import itemgetter
from collections import Counter
from torch.utils.data import Subset

In [9]:
dataset_file = Path("data") / f"medindcls_{args.bertmodel}_{args.doclevel}.json"
if not dataset_file.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, task="MIC", doclevel=args.doclevel, clean=False)
    with open(dataset_file, "wb") as f:
        print(f"Saving dataset under {dataset_file}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)
    
def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

Loading dataset from: data/medindcls_medbert_letter.json


In [12]:
MODELNAME = Path(PRETRAINEDMODEL).stem

GCNNAME = f"{MODELNAME}_{dataset}.pt"
SAVEDIR = Path(f"models/gcn/{args.mixfactor}/{args.doclevel}")
GCNPATH= SAVEDIR / GCNNAME

IGFILE = SAVEDIR / f"ig_attrs_gcn_{MODELNAME}_{args.data}.npz"
SHAPFILE = SAVEDIR / f"shap_values_gcn_{MODELNAME}_{args.data}.npz"

ig_gcn_bert_values = np.load(IGFILE, "rb")['arr_0']
shap_gcn_bert_values = np.load(SHAPFILE, "rb")['arr_0']

In [60]:
pd.DataFrame(ig_gcn_bert_values.sum(0)).describe()

,0
count,2699.000000
mean,-0.083953
std,0.492075
min,-4.008761
25%,-0.334703
50%,-0.043391
75%,0.134982
max,1.706185


In [58]:
(ig_gcn_bert_values > 0).sum()/ig_gcn_bert_values.size

0.4061202365759609

In [14]:
top_n_interpret = 10
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_bert_values = np.argpartition(shap_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]

In [15]:
random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [16]:
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)
top_shap_gcn_bert_values = np.vectorize(map_to_idx)(top_shap_gcn_bert_values)

In [17]:
ig_gcn_bert_df = pd.DataFrame(top_ig_gcn_bert_values, index=test_idx)
shap_gcn_bert_df = pd.DataFrame(top_shap_gcn_bert_values, index=test_idx)

In [18]:
ig_gcn_bert_df = pd.melt(ig_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_bert_df = pd.melt(shap_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [19]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.rel_id])
ig_gcn_bert_df["label"] = labels
ig_gcn_bert_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.rel_id])
shap_gcn_bert_df["label"] = labels
shap_gcn_bert_df["rel_label"] = rel_labels

In [20]:
ig_gcn_bert_source_df = ig_gcn_bert_df[["id", "label"]].drop_duplicates()
ig_gcn_bert_target_df = ig_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_bert_node_df = pd.concat([ig_gcn_bert_source_df, ig_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_bert_source_df = shap_gcn_bert_df[["id", "label"]].drop_duplicates()
shap_gcn_bert_target_df = shap_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_bert_node_df = pd.concat([shap_gcn_bert_source_df, shap_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [21]:
ig_gcn_bert_G=nx.from_pandas_edgelist(ig_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)
shap_gcn_bert_G=nx.from_pandas_edgelist(shap_gcn_bert_df, "id", 'rel_id')#, create_using=nx.DiGraph)

In [67]:
ig_gcn_bert_G.edges(540)

EdgeDataView([(540, 531), (540, 534), (540, 533)])

In [22]:
ig_gcn_bert_id_df = ig_gcn_bert_df[["id", "label"]]
ig_gcn_bert_rel_df = ig_gcn_bert_df[["rel_id", "rel_label"]]
shap_gcn_bert_id_df = shap_gcn_bert_df[["id", "label"]]
shap_gcn_bert_rel_df = shap_gcn_bert_df[["rel_id", "rel_label"]]

In [23]:
new_columns = ["id", "label"]
ig_gcn_bert_id_df.columns = new_columns
ig_gcn_bert_rel_df.columns = new_columns
shap_gcn_bert_id_df.columns = new_columns
shap_gcn_bert_rel_df.columns = new_columns

In [24]:
ig_gcn_bert_id_rel_df = pd.concat([ig_gcn_bert_id_df, ig_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_bert_id2label = dict(zip(ig_gcn_bert_id_rel_df.id, ig_gcn_bert_id_rel_df.label))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2label, "label")

shap_gcn_bert_id_rel_df = pd.concat([shap_gcn_bert_id_df, shap_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_bert_id2label = dict(zip(shap_gcn_bert_id_rel_df.id, shap_gcn_bert_id_rel_df.label))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2label, "label")

In [25]:
ig_gcn_bert_id2text = {id: dataset.texts[id] for id in ig_gcn_bert_id_rel_df.id}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2text, "text")

shap_gcn_bert_id2text = {id: dataset.texts[id] for id in shap_gcn_bert_id_rel_df.id}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2text, "text")

In [26]:
ig_gcn_bert_id2drug = {node: ig_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_bert_G.nodes()}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2drug, "drug")

shap_gcn_bert_id2drug = {node: shap_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_bert_G.nodes()}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2drug, "drug")

In [27]:
(nx.is_connected(ig_gcn_bert_G), nx.number_connected_components(ig_gcn_bert_G), nx.is_connected(shap_gcn_bert_G), nx.number_connected_components(shap_gcn_bert_G))

(False, 19, False, 7)

In [74]:
ig_gcn_bert_components = nx.connected_components(ig_gcn_bert_G)
ig_gcn_bert_largest_component = max(ig_gcn_bert_components, key=len)
ig_gcn_bert_subgraph = ig_gcn_bert_G.subgraph(ig_gcn_bert_largest_component)

shap_gcn_bert_components = nx.connected_components(shap_gcn_bert_G)
shap_gcn_bert_largest_component = max(shap_gcn_bert_components, key=len)
shap_gcn_bert_subgraph = shap_gcn_bert_G.subgraph(shap_gcn_bert_largest_component)

nx.diameter(ig_gcn_bert_subgraph), nx.diameter(shap_gcn_bert_subgraph)

(22, 18)

In [109]:
components = [x for x in nx.connected_components(ig_gcn_bert_G)]
[len(x) for x in components]

[1912, 10, 10, 10, 11, 10, 12, 12, 10, 10, 10, 12, 10, 10, 10, 10, 10, 10, 10]

In [108]:
[Counter([ig_gcn_bert_G.nodes()[x]["drug"] for x in com]).most_common()[0] for com in components]

[('Ramipril', 145),
 ('Ramipril', 3),
 ('Amlopipin', 2),
 ('Isoptin', 2),
 ('Delix', 2),
 ('Sevikar', 2),
 ('Ramipril', 1),
 ('HCT', 3),
 ('Delix', 2),
 ('Beloc', 1),
 ('Torasemid', 2),
 ('Aldactone', 2),
 ('Corifeo', 2),
 ('Beloc', 2),
 ('Concor', 1),
 ('Amlodipin', 3),
 ('Inspra', 2),
 ('Metoprololsuccinat', 1),
 ('Metoprolol', 2)]

In [30]:
ig_gcn_bert_triadic_closure = nx.transitivity(ig_gcn_bert_G)
shap_gcn_bert_triadic_closure = nx.transitivity(shap_gcn_bert_G)
ig_gcn_bert_triadic_closure, shap_gcn_bert_triadic_closure

(0.20949063996517198, 0.18367708241125963)

In [31]:
ig_gcn_bert_degree_dict = dict(ig_gcn_bert_G.degree(ig_gcn_bert_G.nodes()))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_degree_dict, 'degree')
ig_gcn_bert_sorted_degree = sorted(ig_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_degree_dict = dict(shap_gcn_bert_G.degree(shap_gcn_bert_G.nodes()))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_degree_dict, 'degree')
shap_gcn_bert_sorted_degree = sorted(shap_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_sorted_degree[:10], shap_gcn_bert_sorted_degree[:10]

([(1354, 35),
  (632, 33),
  (1372, 31),
  (123, 31),
  (1991, 26),
  (1355, 23),
  (436, 23),
  (373, 22),
  (1990, 22),
  (1837, 21)],
 [(1354, 35),
  (632, 34),
  (1372, 31),
  (123, 31),
  (1355, 26),
  (1991, 24),
  (1990, 22),
  (436, 22),
  (373, 19),
  (88, 19)])

In [32]:
ig_gcn_bert_betweenness_dict = nx.betweenness_centrality(ig_gcn_bert_G)
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_betweenness_dict, 'betweenness')
ig_gcn_bert_sorted_betweenness = sorted(ig_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)


shap_gcn_bert_betweenness_dict = nx.betweenness_centrality(shap_gcn_bert_G)
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_betweenness_dict, 'betweenness')
shap_gcn_bert_sorted_betweenness = sorted(shap_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_sorted_betweenness[:10], shap_gcn_bert_sorted_betweenness[:10]

([(109, 0.09839269169728862),
  (326, 0.06168925391759462),
  (474, 0.05382581533976217),
  (475, 0.05382581533976217),
  (159, 0.051263189945711884),
  (149, 0.04814998538681836),
  (382, 0.04757603095571717),
  (1444, 0.04538245114366873),
  (1991, 0.04409662323326127),
  (123, 0.04359103710084266)],
 [(1354, 0.07434279745073451),
  (334, 0.0500736538447221),
  (490, 0.047908685131070654),
  (333, 0.047293039993454696),
  (308, 0.047114152248241614),
  (326, 0.04618339834002428),
  (1991, 0.045667743035976725),
  (123, 0.041978155065546134),
  (844, 0.0365814193633946),
  (2447, 0.036211257869484696)])

In [33]:
ig_gcn_bert_communities = nx.community.greedy_modularity_communities(ig_gcn_bert_G)
ig_gcn_bert_modularity_dict = {}
for i, c in enumerate(ig_gcn_bert_communities):
    for name in c:
        ig_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_modularity_dict, 'community')

shap_gcn_bert_communities = nx.community.greedy_modularity_communities(shap_gcn_bert_G)
shap_gcn_bert_modularity_dict = {}
for i, c in enumerate(shap_gcn_bert_communities):
    for name in c:
        shap_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_modularity_dict, 'community')

len(ig_gcn_bert_communities), len(shap_gcn_bert_communities)

(59, 49)

In [34]:
([Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

([0.1871345029239766,
  0.26666666666666666,
  0.8269230769230769,
  0.8333333333333334,
  0.6129032258064516,
  0.9090909090909091,
  0.6756756756756757,
  0.75,
  0.8333333333333334,
  0.7419354838709677],
 [0.20552147239263804,
  0.40268456375838924,
  0.32,
  0.43103448275862066,
  0.53,
  0.9381443298969072,
  0.9072164948453608,
  0.5405405405405406,
  0.8472222222222222,
  0.7083333333333334])

In [110]:
([Counter([ig_gcn_bert_G.nodes()[id]["drug"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["drug"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

([0.10526315789473684,
  0.11666666666666667,
  0.125,
  0.10416666666666667,
  0.10752688172043011,
  0.09090909090909091,
  0.12162162162162163,
  0.09722222222222222,
  0.08333333333333333,
  0.11290322580645161],
 [0.0736196319018405,
  0.08053691275167785,
  0.112,
  0.1896551724137931,
  0.11,
  0.13402061855670103,
  0.1134020618556701,
  0.12162162162162163,
  0.09722222222222222,
  0.16666666666666666])

In [35]:
train_count = 0
val_count = 0
test_count = 0

for node in ig_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in ig_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6431054461181923, 0.0, 0.35689455388180763)

In [36]:
train_count = 0
val_count = 0
test_count = 0

for node in shap_gcn_bert_G.nodes():
    if node not in test_idx: 
        continue
    for n in shap_gcn_bert_G.adj[node]:
        if n in train_idx or n in val_idx:
            train_count += 1
        #elif n in val_idx:
        #    val_count += 1
        else:
            test_count += 1

s = train_count + val_count + test_count
train_count/s, val_count/s, test_count/s

(0.6418181818181818, 0.0, 0.35818181818181816)